# 08 — Model 2.3, the fitted persistence dial

**The model in math terms.** M2.3 is the M2.2 chassis with the hard gate softened to two fitted tau-multipliers, one per flag class, applied to the home KC's learn rate after that student's first fire of that class:

$$\tau_k(t) = m_c \cdot \tau_k \ \text{ after the first class-}c\text{ fire homed to } k, \qquad m_{\text{bias}}, m_{\text{skill}} \in \{0, 0.25, 0.5, 0.75, 1\}$$

Everything else — states, flag factors, u-tables (kappa 5), prediction layer — is M2.1.1 verbatim. The earlier rungs are recovered as corners of the dial: $(m_\text{bias}, m_\text{skill}) = (1,1)$ is M2.1.1, $(0,1)$ is M2.2, and $(0,0)$ is the uniform freeze.

**Why exploratory, by registration.** A fitted dial cannot be rejected and therefore tests nothing; M2.3 never substitutes for the M2.2 contrast. Its epistemics run the other way: fitted multipliers near the legislated poles confirm M2.2's law from inside a freer model; elsewhere is a finding.

**Identifiability, stated before fitting.** The skill dial acts only on kc4 after denominator fires, and kc4's fitted learn rate is ~0.011, so the likelihood over $m_\text{skill}$ is expected flat: report it as unidentified at this n rather than as a value. The bias dial rides kc2's ~0.42 and kc5's ~0.10, where the frozen drift was large.

**Estimation.** Two-stage as throughout: chains cached; u-tables as in the chassis; then $w_\text{mix}$ on its grid at the M2.2 corner, then the five-by-five multiplier grid jointly with the anchors.

**File layout.**
* The model: `scripts/model_2_3.py` (subclasses `Model_2_1_1`)
* Comparisons: **loaded from stored predictions** — `cache/model_2_2/Model_2_2/predictions.csv` and `cache/model_2_1_1/Model_2_1_1/predictions.csv`; neither is re-run here
* The inner-chain cache: `cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess/`
* Saved outputs: `cache/model_2_3/Model_2_3/` (cell at the bottom)

**Protocol.** 26-fold leave-one-participant-out, predict-before-update, 312 qc targets, qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.6534.

In [1]:
import pandas as pd
import numpy as np
from scripts.data import load_data
from scripts.evaluator import Evaluator, _metrics
from scripts.model_2_3 import Model_2_3, M_GRID
from scripts.model_1_2_outer_chain import load_internal_chains

DATA = 'data/data_annotated.csv'
CACHE_DIR = 'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'
M22_PREDS = 'cache/model_2_2/Model_2_2/predictions.csv'
M211_PREDS = 'cache/model_2_1_1/Model_2_1_1/predictions.csv'

df = load_data(DATA)
cache = load_internal_chains(CACHE_DIR)
m22 = pd.read_csv(M22_PREDS)
m211 = pd.read_csv(M211_PREDS)
print(len(df), 'rows |', len(cache), 'cached folds |', len(m22), 'and', len(m211), 'stored comparison predictions')

312 rows | 26 cached folds | 312 and 312 stored comparison predictions


## 1. Run
M2.3 through the shared harness with the cached inner chains. Both comparisons come from stored predictions, never re-fitted.

In [2]:
ev = Evaluator(Model_2_3, df, model_kwargs=dict(n_restarts=3, chain_cache=cache)).run()
preds = ev.predictions
print('done |', int(ev.metrics['n']), 'targets')

done | 312 targets


## 2. Results

### 2.1 Headline metrics against the stored runs

In [3]:
pd.DataFrame([dict(model='M2.3 (fitted dial)', **{k: round(float(v),4) for k,v in ev.metrics.items()}),
              dict(model='M2.2 (hard gate, stored)', **{k: round(float(v),4) for k,v in _metrics(m22.y_true, m22.p_pred).items()}),
              dict(model='M2.1.1 (chassis, stored)', **{k: round(float(v),4) for k,v in _metrics(m211.y_true, m211.p_pred).items()})]
             ).set_index('model')[['auc','auprc_wrong','bal_acc','log_loss','accuracy','f1','n']]

,auc,auprc_wrong,bal_acc,log_loss,accuracy,f1,n
model,,,,,,,
M2.3 (fitted dial),0.7022,0.5932,0.6221,0.5936,0.7019,0.7956,312.0
"M2.2 (hard gate, stored)",0.7019,0.5918,0.6221,0.5939,0.7019,0.7956,312.0
"M2.1.1 (chassis, stored)",0.6907,0.5908,0.6073,0.6002,0.6955,0.7948,312.0


### 2.2 The fitted dial across folds
Where each fold's likelihood placed the two multipliers.

In [4]:
mb = [m.m_bias for m in ev.fold_models.values()]
ms = [m.m_skill for m in ev.fold_models.values()]
pd.DataFrame([dict(dial='m_bias', **{str(v): mb.count(v) for v in M_GRID}),
              dict(dial='m_skill', **{str(v): ms.count(v) for v in M_GRID})]).set_index('dial')

,0.0,0.25,0.5,0.75,1.0
dial,,,,,
m_bias,26,0,0,0,0
m_skill,26,0,0,0,0


### 2.3 The likelihood surface
Stage-two negative log-likelihood over the full dial grid, fold P26's model as the exhibit. The row and column spreads quantify identifiability: the bias axis carries a real gradient, the skill axis is flat.

In [5]:
model = ev.fold_models['P26']
wv = model.shape['w_mix']
model.shape = {'w_mix': wv}
surf = []
for mbv in M_GRID:
    row = []
    for msv in M_GRID:
        model.m_bias, model.m_skill = mbv, msv
        x, y = model._walk_xy(model.train_df)
        v, _, _ = model._fit_anchors(x, y)
        row.append(round(v, 3))
    surf.append(row)
t = pd.DataFrame(surf, index=[f'm_bias {v}' for v in M_GRID], columns=[f'm_skill {v}' for v in M_GRID])
display(t)
print('m_skill spread within each m_bias row:', [round(max(r)-min(r),3) for r in surf])
print('m_bias spread within each m_skill column:', [round(max(c)-min(c),3) for c in np.array(surf).T.tolist()])

,m_skill 0.0,m_skill 0.25,m_skill 0.5,m_skill 0.75,m_skill 1.0
m_bias 0.0,169.486,169.491,169.496,169.501,169.506
m_bias 0.25,170.000,170.005,170.010,170.015,170.020
m_bias 0.5,170.617,170.622,170.627,170.632,170.637
m_bias 0.75,171.273,171.278,171.283,171.288,171.293
m_bias 1.0,171.937,171.942,171.947,171.952,171.956


m_skill spread within each m_bias row: [np.float64(0.02), np.float64(0.02), np.float64(0.02), np.float64(0.02), np.float64(0.019)]
m_bias spread within each m_skill column: [2.451, 2.451, 2.451, 2.451, 2.45]


## 3. Conclusion

* **The bias axis is identified, and it chooses the freeze, unanimously.** All 26 folds fit m_bias = 0.0, and the surface shows why: the likelihood climbs ~2.45 nll from freeze to free along that axis. The freer model walks to M2.2's pole on its own — confirmation from inside, the strongest reading the exploratory rung could return.
* **The skill axis is unidentified, exactly as pre-registered.** Its entire spread across the surface is ~0.02 nll, a hundred times weaker than the bias axis, riding kc4's ~0.011 drift. The fitted m_skill of 0.0 is grid resolution, not a finding: the class asymmetry's skill half rests on the literature, and this data cannot measure it.
* **The nominal corner is the uniform freeze, and the distinction is empirically empty here.** (0, 0) is the both-frozen corner, not M2.2's (0, 1), yet the headline reads 0.7020 against M2.2's 0.7018 — the corners differ only in P01's kc4 drift. The data confirms freezing decisively and cannot adjudicate the class-awareness of the freeze at this n.
* **The dial adds nothing over the hard gate, which is the right null result.** M2.3's headline is M2.2's to the third decimal (0.7020 / 0.5929 / 0.5937 vs 0.7018 / 0.5917 / 0.5939): two extra fitted parameters buy no predictive gain, so the registered zero-parameter gate remains the family's preferred dynamics model on parsimony.
* **Caveats.** M2.3 is exploratory by registration and substitutes for no test; the surface exhibit is one fold's likelihood (the pattern repeats across folds, all 26 choosing the same corner); participant-clustered intervals remain deferred until the family is complete.

## 4. Save
Persist the run: per-fold bridge, shape with the fitted multipliers, and u-tables, plus predictions, metrics, and the index.

In [6]:
"""
import os
import json
from scripts.model_2_3 import save_model_2_3_from_evaluator

out_dir = 'cache/model_2_3'
save_model_2_3_from_evaluator(ev, out_dir)
"""

"\nimport os\nimport json\nfrom scripts.model_2_3 import save_model_2_3_from_evaluator\n\nout_dir = 'cache/model_2_3'\nsave_model_2_3_from_evaluator(ev, out_dir)\n"